<a href="https://colab.research.google.com/github/AzizulHakim00/Glaucomma/blob/main/Glaucomma_RimGraphDG_Single_Cell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RimGraph-DG V4.2 — select a T4 GPU runtime and run this ONE cell.
# T4-safe micro-batches are balanced across gradient-accumulation windows.
GLAUCOMMA_OVERRIDES = {
    'manual_data_dir': '',
    'fast_dev_run': False,
    'run_name': 'paper_run_v42',
    'code_revision': 'rimgraph-dg-v4.2-20260803',
    'seeds': [2029],
    'run_global_baseline': True,
    'run_full_model': True,
    'run_optuna': False,
    'optuna_trials': 8,
    'image_size': 320,
    'batch_size': 2,
    'grad_accum': 2,
    'fpn_dim': 128,
    'gradient_checkpointing': True,
    'baseline_epochs': 12,
    'full_epochs': 30,
    'num_workers': 0,
    'n_visual_examples': 2,
}

import hashlib
import json
import traceback
import urllib.request
from pathlib import Path

COMMIT = 'dbf9e4492c3350800773267d9538cd47c5b2978f'
EXPECTED_RAW_SHA256 = '46ba27c7446662460456bc2bab186729c0df1b3e76533ce44f208150208335e2'
ROOT = f'https://raw.githubusercontent.com/AzizulHakim00/Glaucomma/{COMMIT}'
parts = [f'v4_parts/part_{i:02d}.py' for i in range(7)]
raw_code = '\n'.join(
    urllib.request.urlopen(f'{ROOT}/{name}').read().decode('utf-8')
    for name in parts
)
actual_raw = hashlib.sha256(raw_code.encode('utf-8')).hexdigest()
assert actual_raw == EXPECTED_RAW_SHA256, f'V4 raw runner integrity check failed: {actual_raw}'

patch41_source = urllib.request.urlopen(f'{ROOT}/runner_patch_v41.py').read().decode('utf-8')
patch42_source = urllib.request.urlopen(f'{ROOT}/runner_patch_v42.py').read().decode('utf-8')
patch41_ns, patch42_ns = {}, {}
exec(compile(patch41_source, 'runner_patch_v41.py', 'exec'), patch41_ns, patch41_ns)
exec(compile(patch42_source, 'runner_patch_v42.py', 'exec'), patch42_ns, patch42_ns)
code = patch41_ns['apply_v41'](raw_code)
code = patch42_ns['apply_v42'](code)
compile(code, 'rimgraph_dg_v42_single_cell.py', 'exec')

try:
    exec(code, globals(), globals())
except BaseException:
    trace = traceback.format_exc()
    print(trace)
    local_failure = Path('/content/RimGraph_V42_FAILURE_TRACEBACK.txt')
    try:
        local_failure.write_text(trace, encoding='utf-8')
    except Exception:
        pass
    try:
        failure_dir = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v42')
        failure_dir.mkdir(parents=True, exist_ok=True)
        (failure_dir / 'FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
        (failure_dir / 'FAILURE_STATUS.json').write_text(
            json.dumps({
                'status': 'failed',
                'traceback_file': str(failure_dir / 'FAILURE_TRACEBACK.txt'),
            }, indent=2),
            encoding='utf-8',
        )
    except Exception as drive_error:
        print(f'Could not persist failure to Drive: {drive_error}')
    raise


Mounted at /content/drive
Resolved Colab output: /content/Glaucomma_runs/paper_run_v42
Resolved Drive output: /content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v42
Drive write verification: PASSED


## RimGraph-DG V4 configuration

,setting,value
0,kaggle_dataset,arnavjain1/glaucoma-datasets
1,manual_data_dir,
2,sources,"['ORIGA', 'REFUGE', 'G1020']"
3,fold_targets,"['ORIGA', 'REFUGE', 'G1020']"
4,image_size,320
5,num_workers,0
6,canonicalize_laterality,True
7,exclude_cross_source_duplicates,True
8,project_name,RimGraph_DG_V4
9,run_name,paper_run_v42


## Downloading or locating Kaggle dataset

Using Colab cache for faster access to the 'glaucoma-datasets' dataset.
Dataset root: /kaggle/input/glaucoma-datasets


...

## Dataset audit

label,source,Normal,Glaucoma,Total labelled,With any mask,Known laterality,Excluded unlabeled
0,G1020,724,296,1020,1020,0,0
1,ORIGA,482,168,650,650,650,0
2,REFUGE,720,80,800,800,0,400


## Seed 2029 — held-out ORIGA

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            